In [8]:
!pip install -q -U google-genai pandas

In [9]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("ключ підключений")

ключ підключений


In [10]:
import os
import json
import pandas as pd
from google import genai


client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])


# ---------- 1. Читання CSV файлу ----------

df_input = pd.read_csv("problem_texts.csv")

print(df_input.head())
print("кількість текстів:", len(df_input))
print("колонки:", df_input.columns)


# ---------- 2. Перевірка потрібних колонок ----------

if "text" not in df_input.columns:
    print("немає колонки text")

if "my_interpretation" not in df_input.columns:
    print("немає колонки my_interpretation")

if "problem_type" not in df_input.columns:
    print("немає колонки problem_type")


# ---------- 3. Готуємо дані для Gemini ----------

items = []

for i in range(len(df_input)):
    item = {
        "id": i + 1,
        "text": str(df_input.loc[i, "text"]),
        "my_interpretation": str(df_input.loc[i, "my_interpretation"]),
        "problem_type": str(df_input.loc[i, "problem_type"])
    }

    items.append(item)


# ---------- 4. Один запит до Gemini для всіх 50 текстів ----------

prompt = f"""
Analyze emotional sentiment for these texts.

I will give you a list of texts. For each text return:
id
gemini_sentiment: positive, negative, neutral, or mixed
gemini_explanation: one short sentence
possible_error_reason: why AI can make a mistake here

Return only valid JSON array. Do not add markdown.

Texts:
{json.dumps(items, ensure_ascii=False)}
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

answer = response.text.strip()

answer = answer.replace("```json", "")
answer = answer.replace("```", "")
answer = answer.strip()

print("відповідь Gemini отримана")


# ---------- 5. Перетворюємо відповідь у JSON ----------

try:
    gemini_data = json.loads(answer)
except:
    print("не вийшло нормально прочитати JSON")
    print(answer)

    with open("raw_gemini_answer.txt", "w", encoding="utf-8") as f:
        f.write(answer)

    gemini_data = []


# ---------- 6. Створення таблиці результатів ----------

results = []

for i in range(len(df_input)):
    original_id = i + 1

    found = None

    for item in gemini_data:
        if item.get("id") == original_id:
            found = item
            break

    if found is None:
        found = {
            "gemini_sentiment": "unknown",
            "gemini_explanation": "No result",
            "possible_error_reason": "No result"
        }

    row = {
        "text": df_input.loc[i, "text"],
        "problem_type": df_input.loc[i, "problem_type"],
        "my_interpretation": df_input.loc[i, "my_interpretation"],
        "gemini_sentiment": found.get("gemini_sentiment", "unknown"),
        "gemini_explanation": found.get("gemini_explanation", ""),
        "possible_error_reason": found.get("possible_error_reason", "")
    }

    results.append(row)


df = pd.DataFrame(results)


# ---------- 7. Збереження CSV ----------

df.to_csv("gemini_sentiment_results.csv", index=False)

print("результати збережено")
print(df.head())


# ---------- 8. Підрахунок типів проблем ----------

problem_count = df["problem_type"].value_counts()
sentiment_count = df["gemini_sentiment"].value_counts()

print("типи проблемних текстів:")
print(problem_count)

print("\nрезультати Gemini:")
print(sentiment_count)


# ---------- 9. Створення README файлу ----------

with open("README.txt", "w", encoding="utf-8") as f:
    f.write("Практичне заняття 11. Інтеграція Gemini API\n\n")

    f.write("Мета роботи:\n")
    f.write("Перевірити, як Gemini API аналізує емоційне забарвлення текстів і знайти слабкі місця ШІ.\n\n")

    f.write("Що було зроблено:\n")
    f.write("Я створив CSV-файл з 50 проблемними текстами.\n")
    f.write("У файлі є текст, моя інтерпретація настрою та тип проблеми.\n")
    f.write("Потім усі тексти були проаналізовані через Gemini API.\n")
    f.write("Результати Gemini були порівняні з моєю власною інтерпретацією.\n\n")

    f.write("Кількість текстів: " + str(len(df)) + "\n\n")

    f.write("Типи проблемних текстів:\n")
    f.write(str(problem_count) + "\n\n")

    f.write("Розподіл результатів Gemini:\n")
    f.write(str(sentiment_count) + "\n\n")

    f.write("Результати аналізу:\n\n")

    for i in range(len(df)):
        f.write("Текст " + str(i + 1) + ":\n")
        f.write("Текст: " + str(df.loc[i, "text"]) + "\n")
        f.write("Тип проблеми: " + str(df.loc[i, "problem_type"]) + "\n")
        f.write("Моя інтерпретація: " + str(df.loc[i, "my_interpretation"]) + "\n")
        f.write("Результат Gemini: " + str(df.loc[i, "gemini_sentiment"]) + "\n")
        f.write("Пояснення Gemini: " + str(df.loc[i, "gemini_explanation"]) + "\n")
        f.write("Можлива причина помилки: " + str(df.loc[i, "possible_error_reason"]) + "\n\n")

    f.write("Аналіз помилок:\n")
    f.write("Найбільші труднощі для ШІ можуть викликати сарказм, іронія, сленг, непрямий зміст та змішані відгуки.\n")
    f.write("У сарказмі слова часто звучать позитивно, але реальний зміст негативний.\n")
    f.write("У змішаних текстах є і позитивні, і негативні частини, тому моделі важче вибрати один настрій.\n")
    f.write("Сленг також може бути проблемою, бо деякі слова мають не буквальне значення.\n\n")

    f.write("Висновок:\n")
    f.write("Gemini API добре працює з простими текстами, але може помилятися на складних прикладах. ")
    f.write("Особливо важкими є тексти із сарказмом, іронією, культурним контекстом, сленгом і неоднозначністю. ")
    f.write("Щоб покращити результат, можна давати моделі більше контексту, просити пояснення відповіді ")
    f.write("або порівнювати результат з іншими моделями.\n")


                                                text my_interpretation  \
0  Great, my laptop crashed again right before th...          negative   
1  Thanks for ignoring my message for three days,...          negative   
2  Wow, waiting one hour for coffee is exactly wh...          negative   
3  Perfect, the delivery came late and the box wa...          negative   
4  I just love when apps delete my work without s...          negative   

  problem_type  
0      sarcasm  
1      sarcasm  
2      sarcasm  
3      sarcasm  
4      sarcasm  
кількість текстів: 50
колонки: Index(['text', 'my_interpretation', 'problem_type'], dtype='str')
відповідь Gemini отримана
результати збережено
                                                text problem_type  \
0  Great, my laptop crashed again right before th...      sarcasm   
1  Thanks for ignoring my message for three days,...      sarcasm   
2  Wow, waiting one hour for coffee is exactly wh...      sarcasm   
3  Perfect, the delivery came lat

Висновок

У практичній роботі було перевірено роботу Gemini API для аналізу емоційного забарвлення текстів. Для цього був створений CSV-файл з 50 проблемними прикладами. У текстах були сарказм, іронія, сленг, змішаний контекст, непряма критика та неоднозначні фрази.

Gemini API проаналізував усі тексти та визначив їхній настрій: positive, negative, neutral або mixed. За результатами більшість текстів були визначені як negative, також були positive, mixed і neutral.

Під час аналізу стало видно, що найскладнішими для ШІ є тексти із сарказмом та змішаним контекстом. У таких випадках позитивні слова можуть мати негативний зміст, а один текст може містити і хорошу, і погану частину.

Можна зробити висновок, що Gemini API добре працює з простими текстами, але для складних прикладів йому потрібен контекст. Для покращення результатів можна давати моделі більше пояснень, просити її аргументувати відповідь або порівнювати результат з іншими моделями.